# National Statistics
Relative and absolute effectiveness of all protected land in one country, analyzed as one site, step by step with checks and maps along the way.

How this differs from the PA-level pipeline:
- **Protected land**: every PA (not OECMs) that intersects the country boundary, of any size, Terrestrial or Coastal, Inscribed, Adopted, Designated or Established, not a UNESCO-MAB Biosphere Reserve, designated 1–2017, dissolved into one layer. PAs are not clipped; the country is the region of every Earth Engine step.
- **Treatment cells**: a random sample of 1 km land cells fully inside the protected land, sized as for a site.
- **Control cells**: land anywhere in the country at least 10 km from any PA or OECM other than UNESCO-MAB Biosphere Reserve PAs (no 200 km limit).
- **Matching and effectiveness**: the PA-level functions and settings, with 2018 as the start year, since every PA is designated before 2018.

Nothing is saved. New functions are in `src/national_stats/`. Large countries may time out at the treatment cells step, and absolute effectiveness takes two requests per 100 km piece of the country.

In [ ]:
# Select country by ISO3 code (shapeGroup in geoBoundaries ADM0)
country = "CRI"

In [ ]:
from pathlib import Path
import os
import sys
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    ANALYSIS_START_YR,
    ANALYSIS_END_YR,
    BALANCED_SMD_THRESHOLD,
    COVARIATES,
    DRIVER_LABELS,
    EE_CRS_METERS,
    GLC_PALETTE,
    KERNEL_RADIUS_M,
    MIN_TREATMENT_CELLS,
    REPORT_NO_MAX_SMD,
)

from utils.retries import set_request_limits
from cells.get_control_cells import calc_n_control_samples
from cells.get_treatment_cells import calc_n_treatment
from cells.load_all_cells import build_site_cells
from sites.site_selector import bounds_region
from matching.match_cells import match_treatment_control_mdm
from matching.diagnostics import (
    DIAGNOSTIC_COLUMNS,
    site_diagnostics_row,
    build_report_card,
    evaluate_covariate_balance,
)
from relative_effectiveness.metrics_per_cell import (
    RelativeHabitatConditionAnalyzer,
    RelativeHabitatLossAnalyzer,
)
from relative_effectiveness.relative_scores import (
    SCORE_COLS,
    START_HABITAT_COL,
    build_match_df_with_score_diffs,
    regression_adjust_score_diffs,
    aggregate_site_relative_scores,
)
from relative_effectiveness.relative_effectiveness import (
    COLUMNS as SITE_RELATIVE_COLUMNS,
    SCORE_NAMES,
)
from absolute_effectiveness.absolute_effectiveness import (
    DRIVER_COLUMNS,
    N_HABITATS_LOST,
)
from national_stats.country_cells import (
    INTERIOR_SCALE_M,
    load_country,
    national_pas,
    pa_table,
    protected_land,
    interior_land_image,
    sample_treatment_cells,
    control_land_image,
    sample_control_cells,
    extract_country_covariates,
)
from national_stats.country_absolute import (
    COLUMNS as ABSOLUTE_COLUMNS,
    analyze_country,
    country_rasters,
    loss_analyzer,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)
set_request_limits()

## 1. Country and protected areas
PAs that intersect the country boundary (`filterBounds`). Features with another ISO3 are cross-border PAs or neighbors' PAs that reach the border; only their parts inside the country are used.

In [ ]:
country_geom, country_shape = load_country(country)
pas = national_pas(country_geom)
pa_attrs = pa_table(pas)

print(
    f"{country}: {len(pa_attrs)} PA features of {pa_attrs['SITE_ID'].nunique()} sites"
)
print(
    f"Summed GIS_AREA: {pa_attrs['GIS_AREA'].sum():,.0f} km² "
    "(whole PAs, overlaps counted twice)"
)
display(pa_attrs["ISO3"].value_counts().to_frame("features"))
display(
    pa_attrs.groupby(["REALM", "STATUS"])
    .agg(features=("SITE_ID", "size"), gis_area_km2=("GIS_AREA", "sum"))
    .round(0)
)
# Designations with the most area
display(
    pa_attrs.groupby("DESIG_ENG")
    .agg(features=("SITE_ID", "size"), gis_area_km2=("GIS_AREA", "sum"))
    .sort_values("gis_area_km2", ascending=False)
    .head(10)
    .round(0)
)
pa_attrs["STATUS_YR"].plot.hist(
    bins=30, figsize=(7, 3), title="Designation year (STATUS_YR)"
)
plt.show()

In [ ]:
country_outline = ee.FeatureCollection([ee.Feature(country_geom)]).style(
    color="black", fillColor="00000000", width=2
)
protected_viz = {"palette": ["#2b8cbe"]}

Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.addLayer(
    pas.style(color="2b8cbe", fillColor="2b8cbe33", width=1),
    {},
    "PAs designated before 2018",
)
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 2. Dissolve protected land
Every PA is painted onto one 0/1 layer, so overlapping and adjacent PAs merge. A 1 km cell on the covariate grid is interior if all of its 50 m pixels are protected and it is mostly land. Zoom in to see the interior cells.

In [ ]:
protected = protected_land(pas)
interior = interior_land_image(protected)

Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")
Map.addLayer(protected.selfMask(), protected_viz, "Protected land", 1, 0.6)
Map.addLayer(interior.selfMask(), {"palette": ["#f03b20"]}, "Interior land cells", 0)
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 3. Treatment cells
A random sample of the interior land cells whose centers are in the country, sized as for a site (`calc_n_treatment`).

In [ ]:
treatment_cells, n_interior = sample_treatment_cells(country, country_geom, interior)
print(f"Interior land cells: {n_interior:,}")
print(f"Sample size: {calc_n_treatment(n_interior)}")
print(f"Treatment cells sampled: {len(treatment_cells)}")

# Check: every cell fully protected and centered in the country
treatment_fc = geemap.geopandas_to_ee(treatment_cells.to_crs(epsg=4326))
protected_share = (
    protected.reduceRegions(
        collection=treatment_fc,
        reducer=ee.Reducer.mean(),
        crs=EE_CRS_METERS,
        scale=INTERIOR_SCALE_M,
    )
    .aggregate_array("mean")
    .getInfo()
)
in_country = treatment_cells.centroid.to_crs(epsg=4326).within(country_shape)
print(f"Smallest protected share of a cell: {min(protected_share):.3f}")
print(f"Cells centered in the country: {in_country.mean():.1%}")

In [ ]:
Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")
Map.addLayer(protected.selfMask(), protected_viz, "Protected land", 1, 0.6)
Map.addLayer(treatment_fc, {"color": "eb6834"}, "Treatment cells")
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 4. Control cells
Land anywhere in the country at least 10 km from any PA or OECM (all realms, any status) other than UNESCO-MAB Biosphere Reserve PAs, on a 3 km grid, as for a site but without the 200 km limit. As at the PA level, small PAs that contain no 3 km pixel center are not excluded.

In [ ]:
control_cells = sample_control_cells(
    country, country_geom, country_shape, len(treatment_cells)
)
print(
    f"Control cells: {len(control_cells)} "
    f"(requested {calc_n_control_samples(len(treatment_cells))})"
)

# Check: control cells overlapping the protected land (small or thin PAs the mask misses)
control_fc = geemap.geopandas_to_ee(control_cells.to_crs(epsg=4326))
control_protected = (
    protected.reduceRegions(
        collection=control_fc,
        reducer=ee.Reducer.max(),
        crs=EE_CRS_METERS,
        scale=INTERIOR_SCALE_M,
    )
    .aggregate_array("max")
    .getInfo()
)
print(f"Control cells overlapping protected land: {sum(control_protected)}")

In [ ]:
Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.addLayer(protected.selfMask(), protected_viz, "Protected land", 1, 0.6)
Map.addLayer(
    control_land_image(country_shape).clip(country_geom),
    {"palette": ["#fdae61"]},
    "Control land (10+ km from PAs and OECMs)",
    1,
    0.5,
)
Map.addLayer(treatment_fc, {"color": "eb6834"}, "Treatment cells")
Map.addLayer(control_fc, {"color": "2a78d6"}, "Control cells")
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 5. Covariates
Each cell's covariates, country, ecoregion and biome, read from the covariate stack as for a site. `country` is the LSIB country used for matching, so border cells may carry a neighbor's name.

In [ ]:
cells = build_site_cells(country, treatment_cells, control_cells)
cells_df = extract_country_covariates(cells)

display(
    cells_df.groupby(["country", "protected"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "control", 1: "treatment"})
)
n_treat_kept = int((cells_df["protected"] == 1).sum())
if n_treat_kept < MIN_TREATMENT_CELLS:
    print(f"Warning: fewer than {MIN_TREATMENT_CELLS} treatment cells ({n_treat_kept})")

In [ ]:
# Covariate distributions of treatment and control cells before matching
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, cov in zip(axes.flat, COVARIATES):
    for flag, label, color in ((1, "Treatment", "#eb6834"), (0, "Control", "#2a78d6")):
        values = cells_df.loc[cells_df["protected"] == flag, cov]
        ax.hist(values, bins=30, density=True, alpha=0.5, color=color, label=label)
    ax.set_title(cov)
axes.flat[0].legend()
axes.flat[-1].axis("off")
fig.suptitle(f"Covariates before matching, {country}")
fig.tight_layout()
plt.show()

## 6. Matching
Mahalanobis Distance Matching and diagnostics, with the PA-level settings.

In [ ]:
match_df, treat_df, control_df = match_treatment_control_mdm(cells_df)

In [ ]:
row = site_diagnostics_row(match_df, treat_df, cells_df, country)
results_df = pd.DataFrame([row]).reindex(columns=DIAGNOSTIC_COLUMNS)
card = build_report_card(results_df).iloc[0]

print(f"{country}: assessable = {card['assessable']} ({card['reason']})")
n_matched = int(row["n_matched_treat"])
if n_matched < MIN_TREATMENT_CELLS:
    print(f"Warning: fewer than {MIN_TREATMENT_CELLS} matched treatment cells")

print("\nReport card")
display(card.drop(["site_id", "assessable", "reason"]).to_frame("value"))

per_covariate_prefixes = ("abs_smd_after_", "variance_ratio_")
site_cols = [
    col
    for col in DIAGNOSTIC_COLUMNS
    if col != "site_id" and not col.startswith(per_covariate_prefixes)
]
# Counts as whole numbers, everything else to 3 decimals
site_values = results_df.iloc[0][site_cols].astype(float)
print("Diagnostics")
display(
    site_values.map(lambda x: f"{x:.0f}" if x.is_integer() else f"{x:.3f}").to_frame(
        "value"
    )
)

print("Per-covariate diagnostics")
display(
    pd.DataFrame(
        {
            "abs_smd_after": [row[f"abs_smd_after_{cov}"] for cov in COVARIATES],
            "variance_ratio": [row[f"variance_ratio_{cov}"] for cov in COVARIATES],
        },
        index=COVARIATES,
    ).round(3)
)

In [ ]:
# Love plot: absolute SMD per covariate before and after matching
covariate_results = evaluate_covariate_balance(match_df, cells_df)
love = covariate_results.assign(
    before=covariate_results["smd_before"].abs(),
    after=covariate_results["smd_after"].abs(),
).sort_values("before")  # worst pre-matching imbalance at the top
y = range(len(love))

fig, ax = plt.subplots(figsize=(7, 0.45 * len(love) + 2.2), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")

# Thin connector shows how far matching moved each covariate
ax.hlines(y, love["after"], love["before"], color="#d9d8d4", linewidth=1, zorder=1)
ax.scatter(
    love["before"],
    y,
    s=64,
    facecolors="none",
    edgecolors="green",
    linewidths=2,
    label="Before matching",
    zorder=3,
)
ax.scatter(
    love["after"],
    y,
    s=64,
    color="purple",
    edgecolors="purple",
    label="After matching",
    zorder=3,
)

# Reference line: balance threshold (Feng et al. 2022)
ax.axvline(
    BALANCED_SMD_THRESHOLD,
    color="#52514e",
    linestyle="--",
    linewidth=1,
    zorder=0,
    label="Balance threshold",
)

x_max = max(love[["before", "after"]].max().max(), REPORT_NO_MAX_SMD)
ax.set_xlim(-0.03 * x_max, 1.05 * x_max)  # room so points at 0 aren't clipped
ax.set_yticks(list(y))
ax.set_yticklabels(love["covariate"], color="#0b0b0b")
ax.set_xlabel("Absolute standardized mean difference", color="#52514e")
ax.set_title(f"Covariate balance, {country}", color="#0b0b0b", loc="left")
ax.grid(axis="x", color="#ebeae6", linewidth=0.8)
ax.set_axisbelow(True)
ax.tick_params(colors="#52514e", length=0)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d9d8d4")
ax.legend(
    loc="upper left",
    bbox_to_anchor=(0, -0.16),
    ncol=2,
    frameon=False,
    labelcolor="#0b0b0b",
    fontsize=9,
)
plt.tight_layout()
plt.show()

In [ ]:
# Matched cells with their covariates, as MDM_match_cells.py saves them
matched_ids = pd.concat(
    [match_df["treat_cell_id"], match_df["control_cell_id"]]
).unique()
matched_cells_gdf = cells[["cell_ID", "protected", "geometry"]].merge(
    cells_df[["cell_ID", *COVARIATES]], on="cell_ID"
)
matched_cells_gdf = matched_cells_gdf[matched_cells_gdf["cell_ID"].isin(matched_ids)]
matched_cells = geemap.geopandas_to_ee(matched_cells_gdf)
cells_fc = geemap.geopandas_to_ee(cells)

Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")
Map.addLayer(protected.selfMask(), protected_viz, "Protected land", 1, 0.4)
Map.addLayer(
    cells_fc.filter(ee.Filter.eq("protected", 1)),
    {"color": "#f5b49a"},
    "Candidate Treatment Cells",
)
Map.addLayer(
    cells_fc.filter(ee.Filter.eq("protected", 0)),
    {"color": "95bceb"},
    "Candidate Control Cells",
)
Map.addLayer(
    matched_cells.filter(ee.Filter.eq("protected", 1)),
    {"color": "eb6834"},
    "Matched Treatment Cells",
)
Map.addLayer(
    matched_cells.filter(ee.Filter.eq("protected", 0)),
    {"color": "2a78d6"},
    "Matched Control Cells",
)
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 7. Relative effectiveness
Score the matched cells and compare treatment and control cells, as for a site. The start year is 2018: every PA is designated before 2018, and a site's start year is never earlier than 2018.

In [ ]:
START_YR = ANALYSIS_START_YR
# Input images are read within the country and the intactness kernel's reach
inputs_region = bounds_region(country_shape, KERNEL_RADIUS_M)
rasters = country_rasters(inputs_region, START_YR)

relative_loss = RelativeHabitatLossAnalyzer()
relative_condition = RelativeHabitatConditionAnalyzer()
# Score Habitat Extent and Habitat Loss per cell, with the start-habitat
# proportion that weights matched pairs' loss differences, in one pass
scored = relative_loss.calc_area_scores_per_cell(
    rasters["habitat_end"], rasters["loss"], rasters["habitat_start"], matched_cells
)
scored = relative_condition.calc_intactness_score_per_cell(
    rasters["intactness"], scored
)
scored = relative_condition.calc_condition_score_per_cell(scored)

# Download the scores once, so the tables and map below don't recompute them
scored_gdf = ee.data.computeFeatures(
    {"expression": scored, "fileFormat": "GEOPANDAS_GEODATAFRAME"}
).set_crs(epsg=4326)
scored_gdf = scored_gdf[["geometry", "cell_ID", *SCORE_COLS, START_HABITAT_COL]]
scored = geemap.geopandas_to_ee(scored_gdf)
print(f"Scored {len(scored_gdf)} matched cells")

In [ ]:
match_table = match_df.astype({"treat_cell_id": int, "control_cell_id": int})
match_scores = build_match_df_with_score_diffs(scored_gdf, match_table)
# Covariates for each matched cell come from the matched cells
match_scores = regression_adjust_score_diffs(match_scores, matched_cells_gdf)
print(f"Regression adjusted: {match_scores.attrs['regression_adjusted']}")
match_scores[
    [
        "treat_cell_id",
        "control_cell_id",
        "treat_condition_score",
        "control_condition_score",
        "condition_score_diff",
        "condition_score_diff_adj",
        "treat_loss_score",
        "control_loss_score",
        "loss_score_diff",
        "loss_score_diff_adj",
        "pair_habitat_weight",
    ]
].head()

In [ ]:
treat_means, site_scores = aggregate_site_relative_scores(match_scores)
relative_row = {
    "country": country,
    "start_year": START_YR,
    "end_year": ANALYSIS_END_YR,
    **{f"{name}_unadj": site_scores[name] for name in SCORE_NAMES},
    **{f"{name}_adj": site_scores[f"{name}_adj"] for name in SCORE_NAMES},
    "mean_pair_habitat_weight": site_scores["mean_pair_habitat_weight"],
    "regression_adjusted": match_scores.attrs["regression_adjusted"],
}
# Columns of relative_effectiveness.py, with country in place of site_id
relative_results = pd.DataFrame([relative_row]).reindex(
    columns=["country", *SITE_RELATIVE_COLUMNS[1:]]
)

# Loss is weighted by each pair's start-habitat weight; this is the plain mean of those weights
print(
    f"Mean pair habitat weight (relative loss): {site_scores['mean_pair_habitat_weight']:.3f}"
)
pd.DataFrame(
    {
        "unadjusted": [site_scores[n] for n in SCORE_NAMES],
        "adjusted": [site_scores[f"{n}_adj"] for n in SCORE_NAMES],
        "adjustment": [site_scores[f"{n}_adj"] - site_scores[n] for n in SCORE_NAMES],
    },
    index=[f"Relative {n} score" for n in SCORE_NAMES],
).round(3)

In [ ]:
score_palette = ["#d73027", "#fdae61", "#fee08b", "#d9ef8b", "#1a9850"]
score_viz = {"min": 0, "max": 1, "palette": score_palette}


def score_image(fc, prop):
    """Rasterize a numeric per-feature score so it can be displayed with a palette."""
    return fc.reduceToImage(properties=[prop], reducer=ee.Reducer.first())


Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")
Map.addLayer(protected.selfMask(), protected_viz, "Protected land", 0, 0.4)
Map.addLayer(score_image(scored, "extent_score"), score_viz, "Extent Score", 0)
Map.addLayer(score_image(scored, "intactness_score"), score_viz, "Intactness Score", 0)
Map.addLayer(score_image(scored, "condition_score"), score_viz, "Condition Score")
Map.addLayer(score_image(scored, "loss_score"), score_viz, "Loss Score")
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 8. Absolute effectiveness
Habitat condition and loss over every PA pixel in the country, as for a site. The country is cut into 100 km pieces (`site_chunks`), with two requests per piece.

In [ ]:
absolute_row = analyze_country(country, START_YR, country_shape, pas)
# Columns of absolute_effectiveness.py, with country in place of site_id
absolute_results = pd.DataFrame([absolute_row]).reindex(columns=ABSOLUTE_COLUMNS)

print(f"\nCountry: {country}")
print(f"Analysis Period: {START_YR} - {ANALYSIS_END_YR}")
print(f"Protected land area: {absolute_row['site_land_area_km2']:,.0f} km2")
print(f"\nHabitat Extent Score: {absolute_row['extent']:.2f}")
print(f"Habitat Intactness Score: {absolute_row['intactness']:.2f}")
print(f"\nHabitat Condition Score: {absolute_row['condition']:.2f}")
print(f"Habitat Loss Score: {absolute_row['loss']:.2f}")
print("\nOverall habitat loss:")
loss_analyzer.print_habitat_loss_metrics(
    "Total",
    absolute_row["loss_area_km2"],
    absolute_row["loss_pct_habitat"],
    pct_site=absolute_row["loss_pct_site"],
)
print("\nDrivers of habitat loss:")
for class_id, driver in sorted(
    DRIVER_COLUMNS.items(),
    key=lambda item: absolute_row[f"{item[1]}_area_km2"],
    reverse=True,
):
    if absolute_row[f"{driver}_area_km2"]:
        loss_analyzer.print_habitat_loss_metrics(
            DRIVER_LABELS[class_id],
            absolute_row[f"{driver}_area_km2"],
            absolute_row[f"{driver}_pct_habitat"],
            pct_site=absolute_row[f"{driver}_pct_site"],
        )
print(f"\nTop {N_HABITATS_LOST} types of habitat lost:")
for i in range(1, N_HABITATS_LOST + 1):
    label = absolute_row.get(f"habitat_lost_{i}")
    if label:
        loss_analyzer.print_habitat_loss_metrics(
            label,
            absolute_row[f"habitat_lost_{i}_area_km2"],
            absolute_row[f"habitat_lost_{i}_pct_start"],
            reference_label=f"of protected {label.lower()} at start year",
        )

In [ ]:
Map = geemap.Map(height=700)
Map.add_basemap("Esri.WorldGrayCanvas")
Map.add_basemap("Esri.WorldImagery")
Map.addLayer(
    rasters["glc"].select(f"GLC_{ANALYSIS_END_YR}"),
    {"min": 0, "max": 36, "palette": GLC_PALETTE},
    "Global Land Cover",
    0,
)
Map.addLayer(
    rasters["habitat_end"].selfMask().updateMask(protected),
    {"palette": ["#1a9850"]},
    "Habitat Extent",
)
Map.addLayer(
    rasters["intactness"].updateMask(protected),
    score_viz,
    "Habitat Intactness (zoom in)",
    0,
)
Map.addLayer(
    rasters["loss"].selfMask().updateMask(protected),
    {"palette": ["red"]},
    "Habitat Loss",
)
Map.addLayer(
    rasters["driver_class"].selfMask().updateMask(protected),
    {"min": 1, "max": 5, "palette": ["yellow", "red", "orange", "green", "#c51b7d"]},
    "Drivers of habitat loss",
    0,
)
Map.addLayer(country_outline, {}, "Country")
Map.centerObject(country_geom)
Map

## 9. Results
One row each for the country, with the columns of `relative_effectiveness.py` and `absolute_effectiveness.py` (`country` in place of `site_id`).

In [ ]:
display(relative_results.set_index("country").T)
display(absolute_results.set_index("country").T)